### <b>Data Processing for Churn Prediction</b>

This notebook is the initial pipeline for the Kaggle Playground Series S6E3 competition, focusing on **data understanding & manipulation & dynamic feature engineering.** The primary objective is to transform a massive dataset of ~594k records into a refined format optimized for gradient-boosted trees, specifically *CatBoost.*

<b>Key Processing Highlights</b>

**Hierarchical Encoding:** We move beyond simple integers by applying *Ordinal Encoding* to features like (Contract) and (InternetService), transitioning "technology" hierarchies (e.g., Fiber Optic > DSL).

**Domain-Driven Feature Engineering:** We crafted two specialized flags; (InternetOutage) and (PhoneServiceOutage), to distinguish between customers who simply didn't subscribe to a service & those experiencing technical unavailability.

**Categorical Refinement:** Redundant string values (like "No internet service") were mapped to binary negatives after feature extraction to reduce noise and multicollinearity.

**Model-Ready Architecture:** All categorical variables have been processed into either ordinal values or *one-hot encoded* variables, ensuring zero missing values and a consistent schema between training and testing sets.

**Next Step**: Transitioning from data preparation to model architecture and baseline evaluation.

**Source**
> **[How to Perform Ordinal Encoding Using Sklearn](https://www.geeksforgeeks.org/machine-learning/how-to-perform-ordinal-encoding-using-sklearn/)**

In [1]:
# Install the necessary packages
!pip install catboost > /dev/null
!pip install kaggle > /dev/null

# import the libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import the Scikit-learn & CatBoost libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier # Pool

In [2]:
#
from google.colab import files
files.upload()

!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c playground-series-s6e3

Saving kaggle.json to kaggle.json
mkdir: cannot create directory ‘/root/.kaggle’: File exists
100% 14.9M/14.9M [00:00<00:00, 103MB/s] 



In [3]:
!unzip playground-series-s6e3.zip

Archive:  playground-series-s6e3.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [4]:
# Import the raw data from kaggle
Train = pd.read_csv('train.csv')
Test = pd.read_csv('test.csv')
Sample = pd.read_csv('sample_submission.csv')

# Display the shape of the training set
df = Train.copy()

# Capitalize the variables df variable
df.rename(columns={'gender':'Gender','tenure':'Tenure','id':'Id'},inplace=True)
Test.rename(columns={'gender':'Gender','tenure':'Tenure','id':'Id'},inplace=True)

# Check the Shape of the dataset
df.shape

(594194, 21)

In [5]:
# Ensure that everything is working
df.head()

,Id,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [6]:
# Apply One Hot Encoding on the "Payment Method" variable
df = pd.get_dummies(df,columns=['PaymentMethod'],drop_first=False)
# Despite using the four elements can potentially lead to multicolinearity, but the four elements reflect vital info
Test = pd.get_dummies(Test,columns=['PaymentMethod'],drop_first=False)

# Ensure that the values generated are 0s&1s not False and True
convert_cols = ['PaymentMethod_Bank transfer (automatic)','PaymentMethod_Credit card (automatic)',
                'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']
df[convert_cols] = df[convert_cols].replace({True: 1, False: 0})
Test[convert_cols] = Test[convert_cols].replace({True: 1, False: 0})

from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(categories=[['No', 'DSL', 'Fiber optic']])
df['InternetService_encoded'] = encoder.fit_transform(df[['InternetService']])
Test['InternetService_encoded'] = encoder.transform(Test[['InternetService']])

encoder = OrdinalEncoder(categories=[['Female', 'Male']])
df['Gender_encoded'] = encoder.fit_transform(df[['Gender']])
Test['Gender_encoded'] = encoder.transform(Test[['Gender']])

encoder = OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']])
df['Contract_encoded'] = encoder.fit_transform(df[['Contract']])
Test['Contract_encoded'] = encoder.transform(Test[['Contract']])

/tmp/ipykernel_18026/1562939738.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[convert_cols] = df[convert_cols].replace({True: 1, False: 0})
/tmp/ipykernel_18026/1562939738.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Test[convert_cols] = Test[convert_cols].replace({True: 1, False: 0})


In [7]:
# Delete the original variables after transformation
df.drop(['Gender','InternetService','Contract'],axis=1,inplace=True)
Test.drop(['Gender','InternetService','Contract'],axis=1,inplace=True)

In [8]:
# Columns that can contain 'No internet service'
internet_service_cols = ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']

# Create the InternetOutage column: 1 if ANY of the internet service columns have 'No internet service', otherwise 0.
df['InternetOutage'] = (df[internet_service_cols] == 'No internet service').any(axis=1).astype(int)
Test['InternetOutage'] = (Test[internet_service_cols] == 'No internet service').any(axis=1).astype(int)

# Get a list of existing columns
cols = df.columns.tolist()
cols0 = Test.columns.tolist()

# Find the index of 'PhoneService'
phone_service_idx = cols.index('PhoneService')
phone_service_idx = cols0.index('PhoneService')

# Remove 'InternetOutage' from its current position
if 'InternetOutage' in cols:
    cols.remove('InternetOutage')

# Insert 'InternetOutage' after 'PhoneService'
cols.insert(phone_service_idx + 1, 'InternetOutage')

# Reindex the DataFrame with the new column order
df = df[cols]

In [9]:

# Remove 'InternetOutage' from its current position
if 'InternetOutage' in cols0:
    cols0.remove('InternetOutage')

# Insert 'InternetOutage' after 'PhoneService'
cols0.insert(phone_service_idx + 1, 'InternetOutage')

# Reindex the DataFrame with the new column order
Test = Test[cols0]

In [10]:
# Create the PhoneServiceOutage column: 1 if 'MultipleLines' is 'No phone service', otherwise 0.
df['PhoneServiceOutage'] = (df['MultipleLines'] == 'No phone service').astype(int)
Test['PhoneServiceOutage'] = (Test['MultipleLines'] == 'No phone service').astype(int)

# Get a list of existing columns
cols = df.columns.tolist()
cols0 = Test.columns.tolist()

# Find the index of 'PhoneService'
phone_service_idx = cols.index('InternetOutage')
phone_service_idx = cols0.index('InternetOutage')

# Remove 'PhoneServiceOutage' from its current position (if it exists, though it should be new here)
if 'PhoneServiceOutage' in cols:
    cols.remove('PhoneServiceOutage')

# Insert 'PhoneServiceOutage' after 'PhoneService'
cols.insert(phone_service_idx + 1, 'PhoneServiceOutage')

# Reindex the DataFrame with the new column order
df = df[cols]

In [11]:
# Remove 'PhoneServiceOutage' from its current position (if it exists, though it should be new here)
if 'PhoneServiceOutage' in cols0:
    cols0.remove('PhoneServiceOutage')

# Insert 'PhoneServiceOutage' after 'PhoneService'
cols0.insert(phone_service_idx + 1, 'PhoneServiceOutage')

# Reindex the DataFrame with the new column order
Test = Test[cols0]

In [12]:

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('No internet service', 'No')

for col in Test.select_dtypes(include='object').columns:
    Test[col] = Test[col].replace('No internet service', 'No')

In [13]:
# Ensure that this element is replace after creating the new variable
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('No phone service', 'No')

for col in Test.select_dtypes(include='object').columns:
    Test[col] = Test[col].replace('No phone service', 'No')

In [14]:
binary_object_cols = [
    'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn'
]

binary_map = {'Yes': 1, 'No': 0}

for col in binary_object_cols:
  df[col] = df[col].map(binary_map)

In [15]:
binary_object_cols_test = [
    'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

binary_map_test = {'Yes': 1, 'No': 0}

for col in binary_object_cols_test:
  Test[col] = Test[col].map(binary_map_test)

In [16]:
# Fill NA values
df = df.fillna(False)
Test = Test.fillna(False)

In [17]:
# Check the missing values
print(df.isna().sum().sum(), Test.isna().sum().sum())

0 0


In [20]:
# Save the updated data using Push command, this new data is cleaned and ready for deployment!
df.to_csv('train_modified.csv', index=False)
Test.to_csv('test_modified.csv', index=False)

<b>Building and tuning Machine Learning models in the subsequent notebook.</b>